<a href="https://colab.research.google.com/github/sabyapaul/AgenticAI-Lab/blob/main/ElementaryLearningBuddy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1 — Install packages**

In [1]:
!pip install -q -U openai gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.0 MB/s eta 0:00:00


**Cell 1 — Run the complete application**

In [3]:
import os
from typing import Generator

import gradio as gr
from google.colab import userdata
from openai import OpenAI


# ============================================================
# 1. LOAD OPENAI API KEY
# ============================================================

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY was not found. "
        "Add it to Google Colab Secrets using the key icon."
    )

os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=api_key)


# Change this model if your account uses a different model.
MODEL_NAME = "gpt-5-mini"


# ============================================================
# 2. SYSTEM INSTRUCTIONS
# ============================================================

SYSTEM_PROMPT = """
You are a patient, encouraging elementary-school tutor.

Your students are children in grades 1 through 5.

Your responsibilities:

1. Explain lessons using simple, age-appropriate language.
2. Break difficult ideas into small steps.
3. Use familiar examples from school, home, nature, games, and daily life.
4. Never shame a child for making a mistake.
5. Encourage curiosity and confidence.
6. Avoid mature, frightening, political, or inappropriate content.
7. Do not collect personal information from the child.
8. For math, show the calculation step by step.
9. For English, explain grammar, vocabulary, reading, and writing clearly.
10. For science, distinguish observations, explanations, and experiments.
11. When suggesting an experiment, require adult supervision when appropriate.
12. Keep the lesson focused on the selected grade level.

Format every lesson with these sections:

# Lesson Title

## What You Will Learn

## Easy Explanation

## Examples

## Remember This

## Practice Time

Provide five practice questions.

## Answer Key

Place the answers at the end so the child can try independently first.

End with one encouraging sentence.
"""


# ============================================================
# 3. LESSON GENERATOR
# ============================================================

def create_lesson(
    child_name: str,
    grade: str,
    subject: str,
    topic: str,
    lesson_type: str,
    difficulty: str
) -> Generator[str, None, None]:
    """
    Generate an elementary-school lesson and stream it to Gradio.
    """

    child_name = (child_name or "").strip()
    topic = (topic or "").strip()

    if not topic:
        yield "## Please enter a lesson topic."
        return

    student_reference = child_name if child_name else "the student"

    user_prompt = f"""
Create an elementary-school learning activity.

Student name or reference: {student_reference}
Grade level: {grade}
Subject: {subject}
Topic: {topic}
Activity type: {lesson_type}
Difficulty: {difficulty}

Additional instructions:

- Address the student by name only when a name was provided.
- Keep the vocabulary appropriate for {grade}.
- Make the explanation interesting and easy to follow.
- Include concrete examples.
- Include exactly five practice questions.
- Put the complete answer key after the questions.
- For multiple-choice questions, provide four options.
- For math questions, verify every calculation before responding.
- For science activities, mention adult supervision when materials,
  heat, electricity, chemicals, sharp objects, or outdoor exploration
  could present a risk.
"""

    try:
        stream = client.responses.create(
            model=MODEL_NAME,
            instructions=SYSTEM_PROMPT,
            input=user_prompt,
            stream=True
        )

        lesson_text = ""

        for event in stream:
            if event.type == "response.output_text.delta":
                lesson_text += event.delta
                yield lesson_text

        if not lesson_text:
            yield "The lesson could not be generated. Please try again."

    except Exception as error:
        yield (
            "## Lesson generation failed\n\n"
            f"```text\n{error}\n```\n\n"
            "Check your API key, model access, internet connection, "
            "and OpenAI API billing."
        )


# ============================================================
# 4. QUICK TOPIC SUGGESTIONS
# ============================================================

def update_topics(subject: str):
    """
    Update sample topics when the selected subject changes.
    """

    topic_examples = {
        "English": [
            "Nouns, verbs, and adjectives",
            "Writing a good paragraph",
            "Main idea and supporting details",
            "Synonyms and antonyms",
            "Reading comprehension",
            "Past, present, and future tense"
        ],
        "Math": [
            "Addition and subtraction",
            "Multiplication tables",
            "Division with remainders",
            "Fractions",
            "Place value",
            "Word problems",
            "Area and perimeter"
        ],
        "Science": [
            "The solar system",
            "Plant life cycle",
            "States of matter",
            "Weather and seasons",
            "The human body",
            "Food chains",
            "Forces and motion"
        ]
    }

    choices = topic_examples.get(subject, [])

    return gr.Dropdown(
        choices=choices,
        value=choices[0] if choices else None
    )


# ============================================================
# 5. CLEAR FUNCTION
# ============================================================

def clear_form():
    return (
        "",
        "Grade 3",
        "English",
        "Nouns, verbs, and adjectives",
        "Lesson and practice",
        "Standard",
        "## Your lesson will appear here."
    )


# ============================================================
# 6. GRADIO USER INTERFACE
# ============================================================

with gr.Blocks(title="Elementary Learning Buddy") as demo:

    gr.Markdown(
        """
# 🎒 Elementary Learning Buddy

Create personalized **English, Math, and Science** lessons for children
in Grades 1–5.

Choose a grade, subject, topic, and activity type. The AI tutor will
provide a child-friendly explanation, examples, practice questions,
and an answer key.

> A parent, teacher, or responsible adult should review AI-generated
> educational content before a child uses it.
"""
    )

    with gr.Row():

        with gr.Column(scale=2):

            child_name = gr.Textbox(
                label="Child's first name — optional",
                placeholder="e.g. Sachi"
            )

            grade = gr.Dropdown(
                choices=[
                    "Grade 1",
                    "Grade 2",
                    "Grade 3",
                    "Grade 4",
                    "Grade 5"
                ],
                value="Grade 3",
                label="Grade level"
            )

            subject = gr.Radio(
                choices=[
                    "English",
                    "Math",
                    "Science"
                ],
                value="English",
                label="Subject"
            )

            topic = gr.Dropdown(
                choices=[
                    "Nouns, verbs, and adjectives",
                    "Writing a good paragraph",
                    "Main idea and supporting details",
                    "Synonyms and antonyms",
                    "Reading comprehension",
                    "Past, present, and future tense"
                ],
                value="Nouns, verbs, and adjectives",
                label="Lesson topic",
                allow_custom_value=True
            )

            lesson_type = gr.Dropdown(
                choices=[
                    "Lesson and practice",
                    "Homework help",
                    "Quiz",
                    "Step-by-step explanation",
                    "Reading passage and questions",
                    "Fun learning activity",
                    "Review before a test"
                ],
                value="Lesson and practice",
                label="Activity type"
            )

            difficulty = gr.Radio(
                choices=[
                    "Easy",
                    "Standard",
                    "Challenge"
                ],
                value="Standard",
                label="Difficulty"
            )

            with gr.Row():
                generate_button = gr.Button(
                    "✨ Create Lesson",
                    variant="primary"
                )

                clear_button = gr.Button("Clear")

        with gr.Column(scale=3):

            lesson_output = gr.Markdown(
                value="## Your lesson will appear here."
            )

    gr.Markdown(
        """
### Suggested ways to use the app

Ask the child to read the lesson first, complete the questions without
looking at the answers, explain one answer aloud, and ask an adult or
teacher for help when something remains unclear.

**Important:** This is a study-support tool, not a replacement for a
teacher or the child's school curriculum.
"""
    )

    subject.change(
        fn=update_topics,
        inputs=subject,
        outputs=topic
    )

    generate_button.click(
        fn=create_lesson,
        inputs=[
            child_name,
            grade,
            subject,
            topic,
            lesson_type,
            difficulty
        ],
        outputs=lesson_output
    )

    topic.change(
        fn=create_lesson,
        inputs=[
            child_name,
            grade,
            subject,
            topic,
            lesson_type,
            difficulty
        ],
        outputs=lesson_output
    )

    clear_button.click(
        fn=clear_form,
        inputs=[],
        outputs=[
            child_name,
            grade,
            subject,
            topic,
            lesson_type,
            difficulty,
            lesson_output
        ]
    )


demo.queue().launch(
    share=True,
    debug=True,
    theme=gr.themes.Soft()
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://90e091420f13b8bc7e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://90e091420f13b8bc7e.gradio.live
